# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [ ]:
import pandas as pd
 
# ─────────────────────────────────────────────────────────────
# 1. LOAD RAW DATA
# ─────────────────────────────────────────────────────────────
URL = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(URL)
 
print("=" * 60)
print(f"RAW shape: {df.shape}")
print("=" * 60)
 
 
# ─────────────────────────────────────────────────────────────
# 2. COLUMN NAMES
#    - Drop the unnamed index column (artefact from previous save)
#    - Fix typo: 'employmentstatus' → 'employment_status'
#    - Rename 'month' → 'month_number' for clarity
# ─────────────────────────────────────────────────────────────
df.drop(columns=["unnamed:_0"], inplace=True)
 
df.rename(columns={
    "employmentstatus": "employment_status",
    "month": "month_number",
}, inplace=True)
 
print("\nColumns after rename:")
print(df.columns.tolist())
 
 
# ─────────────────────────────────────────────────────────────
# 3. DATA TYPES
#    - effective_to_date → datetime
#    - number_of_open_complaints → int  (was float 0.0–5.0
#      due to earlier fraction-parsing step)
# ─────────────────────────────────────────────────────────────
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], format="%Y-%m-%d")
df["number_of_open_complaints"] = df["number_of_open_complaints"].astype(int)
 
print("\nDtypes after casting:")
print(df.dtypes)
 
 
# ─────────────────────────────────────────────────────────────
# 4. DUPLICATES
#    Customers legitimately appear multiple times (multiple
#    policies), so we only drop *fully* duplicate rows.
# ─────────────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f"\nFully duplicate rows removed: {dupes}")
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
 
 
# ─────────────────────────────────────────────────────────────
# 5. NULL VALUES
#    No nulls present in this dataset, but we apply a robust
#    strategy so the function stays reusable on future data:
#      - Numeric columns  → fill with column median
#      - String columns   → fill with column mode
# ─────────────────────────────────────────────────────────────
print("\nNull counts per column:")
print(df.isnull().sum())
 
num_cols = df.select_dtypes(include="number").columns
str_cols = df.select_dtypes(include="str").columns
 
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
 
for col in str_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
 
 
# ─────────────────────────────────────────────────────────────
# 6. CATEGORICAL CONSISTENCY
#    - Strip whitespace from all string columns
#    - Expand gender codes for readability: M → Male, F → Female
#    - Note: vehicle_type only has one unique value ('A') —
#      flagged as a low-variance column (no action needed)
# ─────────────────────────────────────────────────────────────
for col in str_cols:
    df[col] = df[col].str.strip()
 
df["gender"] = df["gender"].map({"M": "Male", "F": "Female"})
 
print("\nUnique vehicle_type values (low-variance):", df["vehicle_type"].unique().tolist())
 
 
# ─────────────────────────────────────────────────────────────
# 7. DERIVED COLUMN
#    Add month_name for human-readable reporting
# ─────────────────────────────────────────────────────────────
month_map = {
    1: "January",   2: "February",  3: "March",
    4: "April",     5: "May",       6: "June",
    7: "July",      8: "August",    9: "September",
    10: "October",  11: "November", 12: "December",
}
df["month_name"] = df["month_number"].map(month_map)
 
 
# ─────────────────────────────────────────────────────────────
# 8. COLUMN ORDER  (logical grouping)
# ─────────────────────────────────────────────────────────────
df = df[[
    "customer",
    # demographics
    "state", "gender", "education", "marital_status",
    "employment_status", "income", "location_code",
    # policy
    "policy_type", "policy", "coverage",
    "renew_offer_type", "sales_channel",
    "effective_to_date", "month_number", "month_name",
    # financial
    "customer_lifetime_value", "monthly_premium_auto",
    "total_claim_amount",
    # vehicle
    "vehicle_class", "vehicle_size", "vehicle_type",
    # behaviour
    "response", "number_of_open_complaints",
    "number_of_policies", "months_since_last_claim",
    "months_since_policy_inception",
]]
 
print(f"\nCLEAN shape: {df.shape}")
print("\nFirst 3 rows:")
print(df.head(3).to_string())
 
 
# ─────────────────────────────────────────────────────────────
# 9. PIVOT → LONG FORMAT
#    Complaints by policy type & month
# ─────────────────────────────────────────────────────────────
pivot = df.pivot_table(
    values="number_of_open_complaints",
    index="policy_type",
    columns="month_name",
    aggfunc="sum",
).round(2)
 
long_df = (
    pivot
    .reset_index()
    .melt(id_vars="policy_type", var_name="month", value_name="number_of_complaints")
    .sort_values(["month", "policy_type"])
    .reset_index(drop=True)
)
 
print("\n=== Complaints by Policy Type & Month (Long Format) ===")
print(long_df.to_string(index=False))
 
 
# ─────────────────────────────────────────────────────────────
# 10. SAVE
# ─────────────────────────────────────────────────────────────
df.to_csv("marketing_customer_clean.csv", index=False)
long_df.to_csv("complaints_long_format.csv", index=False)
print("\nSaved: marketing_customer_clean.csv & complaints_long_format.csv")
 


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv")

pivot1 = df.pivot_table(
    values="total_claim_amount",
    index="sales_channel",
    aggfunc="sum"
).round(2).sort_values("total_claim_amount", ascending=False)

pivot1.columns = ["total_revenue"]
print(pivot1)

pivot2 = df.pivot_table(
    values="customer_lifetime_value",
    index="education",
    columns="gender",
    aggfunc="mean"
).round(2)

print(pivot2)

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv")

# Step 1: pivot table (wide format) — sum of complaints by policy type × month
pivot = df.pivot_table(
    values="number_of_open_complaints",
    index="policy_type",
    columns="month",
    aggfunc="sum"
).round(2)

# Step 2: melt to long format — one row per (policy_type, month) combination
long_df = pivot.reset_index().melt(
    id_vars="policy_type",
    var_name="month",
    value_name="number_of_complaints"
)

long_df = long_df.sort_values(["month", "policy_type"]).reset_index(drop=True)
print(long_df)